# 08 — Ablation cho paper: Swin-B-384 · MoE hướng D (MILK10k)

Chạy mọi thí nghiệm của 5 bảng trong paper quanh mô hình gốc **M0** = Swin-B-384 · MoE hướng D (gate theo nguồn +
3 head long-tail) · d_model 256 · r5_nobce · clin + derm + metadata (test Dice 0.512). Mỗi thí nghiệm đổi **đúng 1 yếu tố**.

| Bảng | Variant | Câu hỏi |
|---|---|---|
| 2 · Modality | `C` `D` `CD` `CM` `DM` (+ `M0`) | Mỗi nguồn (C = clinical, D = dermoscopy, M = metadata) đóng góp bao nhiêu |
| 3 · Thành phần | `E` (bỏ gate) · `A` (1 head) · `B` (transformer MoE) · `t0` `t1` (1 head τ) · `nosamp` · `aux0` · `md0` · `d1024` | Từng phần thiết kế có giúp không |
| 4 · Backbone | `sw224` + các run của notebook 07 (gắn làm Input) | 224 hay 384, backbone nào |
| 5 · Quyết định | không cần train: ngưỡng theo lớp trên OOF của M0 | Chỉ đổi ngưỡng giúp bao nhiêu |
| 1 · SOTA | M0 ensemble 5 fold (+ ngưỡng theo lớp) | Đứng ở đâu so với các paper |

**Mức ưu tiên** (định nghĩa trong `experiments/stage6_ablation.yaml`, mỗi fold là 1 run riêng, 2 GPU chạy song song):
- **P1** — `M0`, `CD`, `E`, `t0` × **5 fold** = 20 run (~190 phút/run) → khoảng **3 phiên** Kaggle.
- **P2** — 12 variant còn lại × fold 0 → khoảng **1.5 phiên**.

Hết giờ giữa chừng: Save Version, gắn output làm Input, điền `RESTORE`, chạy lại — run đã xong được bỏ qua.
Kết quả: `outputs/ablation_tables/ablation_tables.md` + file nộp ensemble trong `outputs/ablation_tables/submissions/`.

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. Ô *Cấu hình* đã có sẵn link repo và link dữ liệu; chỉ cần sửa `RESTORE` khi chạy tiếp.
3. **Save Version → Save & Run All (Commit)**: notebook chạy nền tối đa 12 giờ, không cần mở trình duyệt.
   Toàn bộ `outputs/` (log, bảng, file nộp) được lưu vào output của version.
4. **Chưa chạy hết lưới** (hết 12 giờ): Add Input → output của version vừa chạy, điền vào `RESTORE`,
   rồi Save Version lần nữa. Các run đã xong được bỏ qua, run dở dang được chạy lại.

Hai GPU được dùng song song: mỗi GPU chạy một run độc lập (`scripts/run_grid.py`). Checkpoint `.pt` bị xoá
sau khi sinh file nộp để không vượt giới hạn 20GB output; `test_probs.csv` được giữ lại để ensemble.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN THÍ NGHIỆM =====================
P1 = ["M0", "CD", "E", "t0"]                                   # bắt buộc, 5 fold
P2 = ["C", "D", "CM", "DM", "A", "B", "t1", "nosamp", "aux0", "md0", "d1024", "sw224"]   # nên có, fold 0
VARIANTS = P1            # P1 | P2 | P1 + P2 | danh sách tự chọn, vd ["M0", "E"]
FOLDS = None             # None = theo yaml (P1: 0–4, P2: 0) | vd [0] để chạy nhanh mọi variant trên fold 0

# Thư mục outputs của notebook 07 (gắn làm Input) -> Bảng 4 backbone, vd "/kaggle/input/<nb07>/fussion/outputs"
REFERENCE = []
# Dice test leaderboard của các file đã nộp -> điền vào bảng. Khoá = tên run hoặc tên thư mục trong submissions/
TEST_SCORES = {
    "swin_base_384__D__r5_nobce__clin+derm": 0.512,
    "vit_base__D__r5_nobce__clin+derm": 0.479,
    # "swin_base_384__abl_M0__cv5": None,
    # "swin_base_384__abl_M0__cv5_thr": None,
}

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Lưới experiments/stage6_ablation.yaml đã đặt sẵn công thức M0 (d_model 256, batch 8 x 8, grad checkpointing, r5_nobce).
# Chỉ đổi ở đây khi muốn áp cho MỌI variant; None = giữ nguyên. Đổi tham số -> đặt RUN_SUFFIX để không trùng tên run.
PARAMS = {
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
    "train.batch_size": None,          # None = 8; CUDA out of memory -> 4 và train.accum_steps 16
    "train.accum_steps": None,
}
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (không cần GPU, ~1 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Danh sách run và thời gian ước tính

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
import yaml
exp = yaml.safe_load(open("experiments/stage6_ablation.yaml", encoding="utf-8"))
select_arg = f"--variants {','.join(VARIANTS)}" + (f" --folds {','.join(map(str, FOLDS))}" if FOLDS else "")

# Phút / run trên T4 (đo ở notebook 07): 2 ảnh 384 ~190, 1 ảnh 384 ~75, Swin-B 224 ~70
def minutes(v):
    spec = exp["variants"][v]
    if "224" in spec.get("config", ""):
        return 70
    return 75 if len(spec.get("views", ["clin", "derm"])) == 1 else 190
n_folds = lambda v: len(FOLDS) if FOLDS else len(exp["variants"][v].get("folds", [0]))
total = sum(minutes(v) * n_folds(v) for v in VARIANTS)
print(f"{sum(n_folds(v) for v in VARIANTS)} run, ~{total / 60 / 2:.1f} giờ trên 2 GPU "
      f"(~{total / 60 / 2 / TIME_BUDGET_H:.1f} phiên {TIME_BUDGET_H} giờ)")
!python scripts/run_grid.py experiments/stage6_ablation.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py experiments/stage6_ablation.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --est-run-h 3.2 --time-budget-h {TIME_BUDGET_H}

## 5. Bảng kết quả
Gộp mọi run trong `outputs` (và `REFERENCE`): mean ± std qua fold, Δ so với M0 ghép cặp theo fold, ensemble test
cho variant ≥ 2 fold, ngưỡng theo lớp cho M0. Nộp các file trong `outputs/ablation_tables/submissions/` lên leaderboard
rồi điền điểm vào `TEST_SCORES` và chạy lại ô này (không cần train lại).

In [ ]:
yaml.safe_dump({k: v for k, v in TEST_SCORES.items() if v is not None}, open("outputs/test_scores.yaml", "w"))
ref_dirs = " ".join(r for r in REFERENCE if os.path.isdir(r))
!python scripts/ablation_tables.py --dirs outputs {ref_dirs} --test-scores outputs/test_scores.yaml > /dev/null
display(Markdown(open("outputs/ablation_tables/ablation_tables.md", encoding="utf-8").read()))

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"08_ablation_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))